In [28]:
import torch
import torch.nn as nn
import torch.optim as optim

In [31]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [32]:
iris = load_iris()

In [35]:
X = iris.data    # features for each flower (sepal len, sepal width etc)
y = iris.target  # labels/ classes  (0: seprosa, 1: versicolr , 2: virginica)

print(X[:5])
print("=====")
print(y[:5])

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
=====
[0 0 0 0 0]


In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # split dataset into 80-20

In [37]:
# feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [38]:
# convert to tensors

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [40]:
# BUild NN

class IrisModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 8)  # 4 i/p, 8 hidden layers
        self.relu = nn.ReLU()     # activation func = ReLU
        self.fc2 = nn.Linear(8, 3)  # 8 hidden layers, 3 o/p ===> 3 flower classes

    def forward(self, x):

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [41]:
model = IrisModel()
print(model)

IrisModel(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=3, bias=True)
)


In [42]:
# loss func
criterion = nn.CrossEntropyLoss()

In [43]:
# optimzer
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [45]:
# training loop

epochs = 100

for epoch in range(epochs):

    outputs = model(X_train)   # fwd pass

    loss = criterion(outputs, y_train)   # loss calc

    optimizer.zero_grad()  # prev gradients clean

    loss.backward()   # bakprop

    optimizer.step()  # weights update

    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 0.1063
Epoch [20/100], Loss: 0.0956
Epoch [30/100], Loss: 0.0872
Epoch [40/100], Loss: 0.0806
Epoch [50/100], Loss: 0.0753
Epoch [60/100], Loss: 0.0709
Epoch [70/100], Loss: 0.0675
Epoch [80/100], Loss: 0.0647
Epoch [90/100], Loss: 0.0623
Epoch [100/100], Loss: 0.0602


In [46]:
with torch.no_grad():

    predictions = model(X_test)

    predicted_classes = torch.argmax(predictions, dim=1)

    accuracy = (predicted_classes == y_test).float().mean()

    print("Accuracy:", accuracy.item())

Accuracy: 1.0
